# Day 072 | LSTM & GRU Networks

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning

**Author:** Ripon Chandra Malo

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| LSTM cell | RNN cell with forget, input, output gates | Selectively remembering long consolidation history |
| Forget gate | Decides what past information to discard | Ignoring irrelevant early-stage noise |
| GRU cell | Simplified gated unit (reset + update gates) | Lighter model for real-time piezometer monitoring |
| `nn.LSTM` / `nn.GRU` | PyTorch gated RNN modules | Building practical sequence models |
| Vanishing gradient solution | Gates preserve gradients over long sequences | Learning from months of pore pressure data |

---

## Our Geotechnical Problem

A soft clay layer is being **preloaded** with a surcharge fill. Piezometers installed at mid-depth
record **excess pore water pressure** that dissipates over time as consolidation proceeds.

We will:
1. Understand why vanilla RNNs fail on long pore pressure sequences
2. Learn how LSTM gates solve the vanishing gradient problem
3. See how GRU simplifies the gating mechanism
4. Build and compare RNN vs LSTM vs GRU on pore pressure prediction

---

In [ ]:
# ───────────────────────────────────────────────────────
# IMPORTS
# ───────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 1: The Vanishing Gradient Problem — Why We Need LSTM

Recall from Day 071: a vanilla RNN passes a hidden state through time.
At each step the hidden state is multiplied by a weight matrix and squashed by `tanh`.

Over many time steps, gradients are **multiplied repeatedly** by the same weight matrix.
If the largest singular value is < 1, gradients **vanish** exponentially.

For pore pressure dissipation that takes **200+ days**, a vanilla RNN simply cannot learn
the relationship between early loading and late-stage drainage.

---

In [ ]:
# ───────────────────────────────────────────────────────
# GENERATE SYNTHETIC PORE PRESSURE DISSIPATION DATA
# ───────────────────────────────────────────────────────
# Excess pore pressure follows: u(t) = u0 * exp(-t/tau) + noise
# Multiple curves simulate different soil layers / loading stages

def generate_pore_pressure_data(n_samples=50, seq_len=200, dt=1.0):
    """
    Generate synthetic pore water pressure dissipation curves.
    
    Parameters:
    -----------
    n_samples : int - number of dissipation curves
    seq_len   : int - number of time steps (days)
    dt        : float - time step (days)
    
    Returns:
    --------
    data : np.array of shape (n_samples, seq_len, 1)
    """
    t = np.arange(seq_len) * dt
    data = np.zeros((n_samples, seq_len, 1))
    
    for i in range(n_samples):
        u0 = np.random.uniform(80, 150)          # initial excess PWP (kPa)
        tau = np.random.uniform(30, 80)           # time constant (days)
        noise_std = np.random.uniform(1.0, 3.0)   # sensor noise
        
        u = u0 * np.exp(-t / tau) + np.random.randn(seq_len) * noise_std
        data[i, :, 0] = u
    
    return data

raw_data = generate_pore_pressure_data(n_samples=100, seq_len=200)
print(f"Generated data shape: {raw_data.shape}")
print(f"  {raw_data.shape[0]} dissipation curves")
print(f"  {raw_data.shape[1]} time steps each")
print(f"  {raw_data.shape[2]} feature (pore pressure)")

In [ ]:
# ───────────────────────────────────────────────────────
# VISUALIZE SAMPLE DISSIPATION CURVES
# ───────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(8):
    ax.plot(raw_data[i, :, 0], alpha=0.7, label=f'Sample {i+1}')

ax.set_xlabel('Time (days)')
ax.set_ylabel('Excess Pore Water Pressure (kPa)')
ax.set_title('Synthetic Pore Pressure Dissipation Curves')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────
# PREPARE SEQUENCES: INPUT (t=0..T-2) -> TARGET (t=1..T-1)
# ───────────────────────────────────────────────────────

# Normalize data
data_mean = raw_data.mean()
data_std = raw_data.std()
data_norm = (raw_data - data_mean) / data_std

X = data_norm[:, :-1, :]   # input:  steps 0 to T-2
y = data_norm[:, 1:, :]    # target: steps 1 to T-1

# Train / test split
n_train = 80
X_train = torch.FloatTensor(X[:n_train])
y_train = torch.FloatTensor(y[:n_train])
X_test  = torch.FloatTensor(X[n_train:])
y_test  = torch.FloatTensor(y[n_train:])

train_ds = TensorDataset(X_train, y_train)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")

## Part 2: LSTM — Long Short-Term Memory

An LSTM cell has **three gates** that control information flow:

| Gate | Formula | Purpose |
|------|---------|--------|
| **Forget** | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | What to erase from cell state |
| **Input**  | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | What new info to store |
| **Output** | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | What to output from cell state |

The **cell state** $C_t$ acts as a conveyor belt:

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
$$h_t = o_t \odot \tanh(C_t)$$

**Geotechnical intuition:** The forget gate might learn to discard short-term sensor noise,
while the cell state preserves the long-term consolidation trend.

---

In [ ]:
# ───────────────────────────────────────────────────────
# MANUAL LSTM CELL — UNDERSTANDING THE GATES
# ───────────────────────────────────────────────────────

class ManualLSTMCell(nn.Module):
    """Hand-coded LSTM cell to see each gate explicitly."""
    
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        
        # Combined weight matrix for all 4 transformations
        self.W = nn.Linear(input_size + hidden_size, 4 * hidden_size)
    
    def forward(self, x_t, h_prev, c_prev):
        """
        x_t:    (batch, input_size)
        h_prev: (batch, hidden_size)
        c_prev: (batch, hidden_size)
        """
        combined = torch.cat([h_prev, x_t], dim=1)
        gates = self.W(combined)
        
        # Split into 4 components
        i_gate = torch.sigmoid(gates[:, :self.hidden_size])                       # Input gate
        f_gate = torch.sigmoid(gates[:, self.hidden_size:2*self.hidden_size])      # Forget gate
        c_cand = torch.tanh(gates[:, 2*self.hidden_size:3*self.hidden_size])       # Cell candidate
        o_gate = torch.sigmoid(gates[:, 3*self.hidden_size:])                     # Output gate
        
        # Update cell state and hidden state
        c_new = f_gate * c_prev + i_gate * c_cand
        h_new = o_gate * torch.tanh(c_new)
        
        return h_new, c_new, {'forget': f_gate, 'input': i_gate, 'output': o_gate}

# Quick test
cell = ManualLSTMCell(input_size=1, hidden_size=8)
x_single = torch.randn(2, 1)
h0 = torch.zeros(2, 8)
c0 = torch.zeros(2, 8)
h1, c1, gate_vals = cell(x_single, h0, c0)
print(f"Input shape:  {x_single.shape}")
print(f"Hidden shape: {h1.shape}")
print(f"Cell shape:   {c1.shape}")
print(f"Forget gate (sample 0): {gate_vals['forget'][0, :4].detach().numpy().round(3)}")

## Part 3: GRU — Gated Recurrent Unit

GRU simplifies LSTM by merging the forget and input gates into a single **update gate**,
and combining cell state and hidden state:

| Gate | Formula | Purpose |
|------|---------|--------|
| **Reset** | $r_t = \sigma(W_r [h_{t-1}, x_t])$ | How much past to forget |
| **Update** | $z_t = \sigma(W_z [h_{t-1}, x_t])$ | Balance between old and new |

$$\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t])$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

**Key differences from LSTM:**
- **Fewer parameters** (2 gates vs 3 gates, no separate cell state)
- **Faster training** — good for real-time monitoring applications
- **Similar performance** on many geotechnical sequence tasks

---

In [ ]:
# ───────────────────────────────────────────────────────
# BUILD THREE MODELS: RNN vs LSTM vs GRU
# ───────────────────────────────────────────────────────

class SequenceModel(nn.Module):
    """Generic wrapper for RNN / LSTM / GRU comparison."""
    
    def __init__(self, model_type='LSTM', input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.model_type = model_type
        self.hidden_size = hidden_size
        
        if model_type == 'RNN':
            self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        elif model_type == 'GRU':
            self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, _ = self.rnn(x)          # out: (batch, seq_len, hidden)
        predictions = self.fc(out)     # (batch, seq_len, 1)
        return predictions

# Count parameters for each model
for mtype in ['RNN', 'LSTM', 'GRU']:
    model = SequenceModel(mtype, input_size=1, hidden_size=32)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{mtype:5s}: {n_params:,} parameters")

In [ ]:
# ───────────────────────────────────────────────────────
# TRAINING FUNCTION
# ───────────────────────────────────────────────────────

def train_model(model, train_dl, X_test, y_test, epochs=60, lr=0.005):
    """
    Train a sequence model and record losses.
    
    Returns:
    --------
    train_losses, test_losses : lists of loss per epoch
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_losses = []
    test_losses = []
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in train_dl:
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
        
        train_losses.append(epoch_loss / len(train_dl))
        
        model.eval()
        with torch.no_grad():
            test_pred = model(X_test)
            test_loss = criterion(test_pred, y_test).item()
            test_losses.append(test_loss)
        
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs} | Train: {train_losses[-1]:.6f}  Test: {test_losses[-1]:.6f}")
    
    return train_losses, test_losses

In [ ]:
# ───────────────────────────────────────────────────────
# TRAIN ALL THREE MODELS
# ───────────────────────────────────────────────────────

results = {}

for mtype in ['RNN', 'LSTM', 'GRU']:
    print(f"\n{'='*50}")
    print(f"Training {mtype}...")
    print(f"{'='*50}")
    
    torch.manual_seed(42)
    model = SequenceModel(mtype, input_size=1, hidden_size=32)
    train_l, test_l = train_model(model, train_dl, X_test, y_test, epochs=60)
    results[mtype] = {'model': model, 'train': train_l, 'test': test_l}

In [ ]:
# ───────────────────────────────────────────────────────
# COMPARE TRAINING CURVES
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'RNN': '#e74c3c', 'LSTM': '#2980b9', 'GRU': '#27ae60'}

for mtype in ['RNN', 'LSTM', 'GRU']:
    axes[0].plot(results[mtype]['train'], label=mtype, color=colors[mtype], linewidth=2)
    axes[1].plot(results[mtype]['test'],  label=mtype, color=colors[mtype], linewidth=2)

axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Test Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('RNN vs LSTM vs GRU  |  Pore Pressure Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 4: Visualizing Predictions

Let us compare how each model predicts pore pressure dissipation on unseen test curves.
We expect LSTM and GRU to better capture the long-term exponential decay.

---

In [ ]:
# ───────────────────────────────────────────────────────
# VISUALIZE PREDICTIONS ON TEST SAMPLES
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, ax in enumerate(axes):
    # True values (denormalize)
    true_vals = y_test[idx, :, 0].numpy() * data_std + data_mean
    ax.plot(true_vals, 'k-', linewidth=2, label='True', alpha=0.8)
    
    for mtype in ['RNN', 'LSTM', 'GRU']:
        model = results[mtype]['model']
        model.eval()
        with torch.no_grad():
            pred = model(X_test[idx:idx+1])
        pred_vals = pred[0, :, 0].numpy() * data_std + data_mean
        ax.plot(pred_vals, '--', color=colors[mtype], linewidth=1.5, label=mtype)
    
    ax.set_title(f'Test Sample {idx+1}')
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Excess PWP (kPa)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Pore Pressure Predictions: RNN vs LSTM vs GRU', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 5: Gradient Flow Comparison

The key advantage of LSTM and GRU is maintaining gradient flow over long sequences.
Let us measure how well each architecture propagates gradients from the last time step
back to the first.

---

In [ ]:
# ───────────────────────────────────────────────────────
# GRADIENT FLOW COMPARISON
# ───────────────────────────────────────────────────────

def measure_gradient_flow(model_type, seq_lengths=[20, 50, 100, 150, 199]):
    """Measure gradient magnitude at the first time step for varying sequence lengths."""
    grad_norms = []
    
    for sl in seq_lengths:
        torch.manual_seed(42)
        model = SequenceModel(model_type, input_size=1, hidden_size=32)
        
        x = X_train[0:1, :sl, :].clone().requires_grad_(True)
        pred = model(x)
        
        # Loss only on the last prediction
        loss = pred[0, -1, 0] ** 2
        loss.backward()
        
        # Gradient at first time step
        grad_norm = x.grad[0, 0, :].norm().item()
        grad_norms.append(grad_norm)
    
    return seq_lengths, grad_norms

fig, ax = plt.subplots(figsize=(10, 5))
for mtype in ['RNN', 'LSTM', 'GRU']:
    lengths, grads = measure_gradient_flow(mtype)
    ax.plot(lengths, grads, 'o-', color=colors[mtype], linewidth=2, markersize=8, label=mtype)

ax.set_xlabel('Sequence Length (days)')
ax.set_ylabel('Gradient Norm at t=0')
ax.set_title('Gradient Flow Through Time  |  Vanishing Gradient Comparison')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────
# INSPECT LSTM GATE ACTIVATIONS OVER TIME
# ───────────────────────────────────────────────────────

lstm_model = results['LSTM']['model']
lstm_model.eval()

sample = X_test[0:1]  # (1, 199, 1)
seq_len_s = sample.shape[1]
hs = lstm_model.hidden_size

# Step through one time step at a time to observe gates
lstm_layer = lstm_model.rnn
h_t = torch.zeros(1, 1, hs)
c_t = torch.zeros(1, 1, hs)

h_norms = []
c_norms = []

with torch.no_grad():
    for t in range(seq_len_s):
        x_t = sample[:, t:t+1, :]
        _, (h_t, c_t) = lstm_layer(x_t, (h_t, c_t))
        h_norms.append(h_t.squeeze().norm().item())
        c_norms.append(c_t.squeeze().norm().item())

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(h_norms, color='#2980b9', linewidth=1.5)
axes[0].set_ylabel('Hidden State Norm')
axes[0].set_title('LSTM Internal States Over Time (Test Sample 1)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(c_norms, color='#e74c3c', linewidth=1.5)
axes[1].set_ylabel('Cell State Norm')
axes[1].set_xlabel('Time Step (days)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────
# FINAL COMPARISON TABLE
# ───────────────────────────────────────────────────────

print("\n" + "="*65)
print("MODEL COMPARISON SUMMARY")
print("="*65)
print(f"{'Model':<8} {'Params':>8} {'Final Train MSE':>16} {'Final Test MSE':>16}")
print("-"*65)

for mtype in ['RNN', 'LSTM', 'GRU']:
    model = results[mtype]['model']
    n_params = sum(p.numel() for p in model.parameters())
    train_mse = results[mtype]['train'][-1]
    test_mse = results[mtype]['test'][-1]
    print(f"{mtype:<8} {n_params:>8,} {train_mse:>16.6f} {test_mse:>16.6f}")

print("="*65)

## Practice Exercises

Try these exercises to deepen your understanding:

```python
# Exercise 1: Multi-layer LSTM
# Build an LSTM with num_layers=2 and compare performance.
# Does adding depth help for this problem?
# model_deep = SequenceModel('LSTM', input_size=1, hidden_size=32, num_layers=2)

# Exercise 2: Bidirectional LSTM
# Modify SequenceModel to use bidirectional=True in nn.LSTM.
# Note: output size doubles! Adjust the Linear layer accordingly.

# Exercise 3: Multiple features
# Generate data with TWO features: pore pressure + settlement.
# Modify input_size=2 and train an LSTM to predict both.

# Exercise 4: Longer sequences
# Generate data with seq_len=500 and compare RNN vs LSTM.
# Does the advantage of LSTM grow with sequence length?
```

---

## Key Takeaways

1. **LSTM solves vanishing gradients** through a cell state that acts as a memory conveyor belt
2. **Three gates** (forget, input, output) control what to remember, update, and output
3. **GRU is a lighter alternative** with two gates (reset, update) and fewer parameters
4. **For pore pressure prediction**, both LSTM and GRU significantly outperform vanilla RNN
5. **Gradient flow** through LSTM/GRU remains stable even for 200+ step sequences

---

## How This Connects

| ML Concept | Geotechnical Application |
|-----------|------------------------|
| LSTM forget gate | Ignoring transient sensor noise in piezometer data |
| GRU update gate | Balancing recent vs historical monitoring readings |
| Cell state | Preserving long-term consolidation trend information |
| Gradient clipping | Stabilizing training with noisy field data |
| Model comparison | Choosing the right architecture for monitoring deployment |